In [1]:
from pathlib import Path
import time
import pandas as pd
from IPython.display import display, clear_output

BASE = Path("D:/adaptive_federated_gnn")
INPUT_PATH = BASE / "outputs/day20/three_model_predictions.csv"
OUT = BASE / "outputs/day22"
OUT.mkdir(parents=True, exist_ok=True)

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Missing predictions file: {INPUT_PATH}. "
        "Complete Day 20 first."
    )

predictions = pd.read_csv(INPUT_PATH)

print("Predictions loaded:", len(predictions))
print("Columns:", predictions.columns.tolist())
display(predictions.head())

Predictions loaded: 85829
Columns: ['edge_index', 'actual_label', 'standard_fedavg_prediction', 'standard_fedavg_attack_probability', 'adaptive_fedavg_prediction', 'adaptive_fedavg_attack_probability', 'private_fedavg_prediction', 'private_fedavg_attack_probability']


,edge_index,actual_label,standard_fedavg_prediction,standard_fedavg_attack_probability,adaptive_fedavg_prediction,adaptive_fedavg_attack_probability,private_fedavg_prediction,private_fedavg_attack_probability
0,2,0,0,0.459778,0,0.460461,1,0.994819
1,5,0,0,0.457212,0,0.457916,1,0.998642
2,14,0,0,0.466843,0,0.467314,1,0.998379
3,15,0,0,0.480987,0,0.482132,1,0.999892
4,16,0,0,0.431740,0,0.432447,1,0.986732


In [2]:
MODEL_PREFIX = "adaptive_fedavg"

prediction_col = f"{MODEL_PREFIX}_prediction"
probability_col = f"{MODEL_PREFIX}_attack_probability"

required_columns = [
    "edge_index", "actual_label",
    prediction_col, probability_col
]

missing = [c for c in required_columns if c not in predictions.columns]

if missing:
    raise ValueError(f"Missing columns in predictions file: {missing}")

print("Selected model:", MODEL_PREFIX)
print("Ready to simulate detection.")

Selected model: adaptive_fedavg
Ready to simulate detection.


In [3]:
# Process a maximum of 100 records for a quick demonstration
stream_data = predictions.head(100).copy()

detection_log = []

for _, row in stream_data.iterrows():
    edge_id = int(row["edge_index"])
    predicted_label = int(row[prediction_col])
    attack_probability = float(row[probability_col])

    status = "ATTACK" if predicted_label == 1 else "BENIGN"

    record = {
        "timestamp": pd.Timestamp.now().isoformat(timespec="seconds"),
        "edge_index": edge_id,
        "prediction": status,
        "attack_probability": round(attack_probability, 4),
        "actual_label_offline": int(row["actual_label"])
    }

    detection_log.append(record)

    # Show the most recent detection
    clear_output(wait=True)
    print("NETWORK ANOMALY DETECTION — SIMULATION")
    print("-" * 45)
    print("Processed records:", len(detection_log), "/", len(stream_data))
    print("Edge index:", edge_id)
    print("Prediction:", status)
    print(f"Attack probability: {attack_probability:.2%}")

    if predicted_label == 1:
        print("ALERT: Potential malicious traffic detected.")
    else:
        print("Status: Traffic classified as benign.")

    time.sleep(0.1)

print("\nSimulation completed.")

NETWORK ANOMALY DETECTION — SIMULATION
---------------------------------------------
Processed records: 100 / 100
Edge index: 307
Prediction: BENIGN
Attack probability: 43.94%
Status: Traffic classified as benign.

Simulation completed.


In [4]:
log_df = pd.DataFrame(detection_log)

output_path = OUT / "detection_log.csv"
log_df.to_csv(output_path, index=False)

print("Detection log saved:", output_path)
print("Total records:", len(log_df))
print("Predicted attacks:", (log_df["prediction"] == "ATTACK").sum())
print("Predicted benign:", (log_df["prediction"] == "BENIGN").sum())

display(log_df.head(10))

Detection log saved: D:\adaptive_federated_gnn\outputs\day22\detection_log.csv
Total records: 100
Predicted attacks: 5
Predicted benign: 95


,timestamp,edge_index,prediction,attack_probability,actual_label_offline
0,2026-10-22T21:38:59,2,BENIGN,0.4605,0
1,2026-10-22T21:38:59,5,BENIGN,0.4579,0
2,2026-10-22T21:38:59,14,BENIGN,0.4673,0
3,2026-10-22T21:38:59,15,BENIGN,0.4821,0
4,2026-10-22T21:38:59,16,BENIGN,0.4324,0
5,2026-10-22T21:38:59,18,BENIGN,0.4360,0
6,2026-10-22T21:39:00,20,ATTACK,0.5321,0
7,2026-10-22T21:39:00,22,BENIGN,0.4617,0
8,2026-10-22T21:39:00,26,BENIGN,0.4342,0
9,2026-10-22T21:39:00,28,BENIGN,0.3170,0


In [5]:
from sklearn.metrics import classification_report, confusion_matrix

actual = log_df["actual_label_offline"]
predicted = (log_df["prediction"] == "ATTACK").astype(int)

print("Classification Report:")
print(classification_report(
    actual, predicted,
    labels=[0, 1],
    target_names=["Benign", "Attack"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(actual, predicted, labels=[0, 1]))

Classification Report:
              precision    recall  f1-score   support

      Benign       1.00      0.95      0.97       100
      Attack       0.00      0.00      0.00         0

    accuracy                           0.95       100
   macro avg       0.50      0.47      0.49       100
weighted avg       1.00      0.95      0.97       100

Confusion Matrix:
[[95  5]
 [ 0  0]]


In [6]:
assert output_path.exists(), "Detection log was not saved."
assert len(log_df) > 0, "No records were processed."

print("DAY 22 COMPLETED")
print("Detection log:", output_path)
print("Records processed:", len(log_df))
print("File size (bytes):", output_path.stat().st_size)

DAY 22 COMPLETED
Detection log: D:\adaptive_federated_gnn\outputs\day22\detection_log.csv
Records processed: 100
File size (bytes): 4125
